# 02 · Pseudobulks — patches and segmented cells, per donor × niche and per donor × cell type

**What this notebook does.** Sums raw counts into the units that every scoring notebook (03–05)
reads, and stores them as small h5ad files. One streaming pass over the 3.5 M-cell object builds all
cell-side pseudobulks; the patch-side ones come from the object built in notebook 01.

| file (`data/xenium/pseudobulk_final/`) | unit | rows |
|---|---|---|
| `pb_patches_morpho_niche_l2.h5ad`, `..._l3.h5ad` | transcript patches, donor × morphology niche | ~50 / ~80 |
| `pb_cells_morpho_niche_l2.h5ad`, `..._l3.h5ad` | segmented cells, donor × morphology niche | ~50 / ~80 |
| `pb_cells_taa_l2_upd.h5ad`, `pb_cells_taa_l3_upd.h5ad` | segmented cells, donor × cell type | ~180 / ~300 |
| `pb_fbcells_morpho_niche_l2.h5ad`, `..._l3.h5ad` | **fibroblast-compartment cells only** (`compart == FbAdipo`, updated annotation), donor × niche | ~45 / ~70 |

Every file carries the same `obs` schema — `donor`, `group`, `level`, `representation`, `age_group`
(epaed / lpaed / adult), `age_group2` (young / adult), `age_years`, `psbulk_cells` (cells or
patches summed), `psbulk_counts` — and a `var['shared_panel']` flag marking the **4,993 genes
measured in every section and in both representations**. Scoring notebooks subset to that flag.

**Annotation.** Cell types are `taa_l2_upd` / `taa_l3_upd` from
`ThyAge_xenium_annos_Fb_upd-clean_2026-09-02.csv` — the `_comb` levels with the updated
fibroblast / adipocyte labels (and ~17 k cells the update removed). Morphology niches come from the
per-cell `patchID`, i.e. each cell inherits the niche of the H&E patch it sits in.

**Filters.** Cells: donor in the seven, patch outside the artifact mask (`morpho_niche_keep`), a
cell-type call. Groups with < 25 cells (or < 25 patches / < 50 k counts for patches) are dropped.

In [1]:
import os
import glob
import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import scipy.sparse as sp
import h5py
import anndata as ad
from anndata.io import sparse_dataset
import decoupler as dc

warnings.filterwarnings('ignore', category=FutureWarning)
today = datetime.now().strftime('%Y-%m-%d')

REPO = '/nfs/team205/vk8/projects/thymus_ageing_atlas'
SPATIAL = f'{REPO}/Spatial_analyses'

CELL_H5AD = f'{SPATIAL}/data/xenium/xenium_proseg/xenium_aging_labels.h5ad'
ANNO_CSV = f'{SPATIAL}/data/xenium/ThyAge_xenium_annos_Fb_upd-clean_2026-09-02.csv'
MORPHO_CSV = f'{SPATIAL}/data/xenium/thyAging_xenium_morpho_niches.csv'
PATCH_H5AD = sorted(glob.glob(f'{SPATIAL}/data/xenium/patchExpr/thyAging_xenium_patchExpr_7donors_*.h5ad'))[-1]

PB_DIR = f'{SPATIAL}/data/xenium/pseudobulk_final'
RESULTS = f'{SPATIAL}/results/xenium/final/02_pseudobulks'
for d in (PB_DIR, RESULTS):
    os.makedirs(d, exist_ok=True)

DONOR_AGE = {'U51': 'epaed', 'U48': 'epaed', 'Z16': 'lpaed', 'Z17': 'lpaed', 'A67': 'adult', 'A72': 'adult', 'A70': 'adult'}
DONOR_AGE_YEARS = {'U51': 0.04, 'U48': 0.5, 'Z16': 8.0, 'Z17': 11.0, 'A67': 20.0, 'A72': 33.0, 'A70': 37.0}
DONOR_ORDER = ['U51', 'U48', 'Z16', 'Z17', 'A67', 'A72', 'A70']
AGE_ORDER = ['epaed', 'lpaed', 'adult']
AGE2 = {'epaed': 'young', 'lpaed': 'young', 'adult': 'adult'}
MORPHO_L2_ORDER = ['cortex_septal', 'cortex', 'medulla', 'medulla_hc', 'adipose', 'fibrosis_adipose', 'fibrosis', 'vessel']
MORPHO_L3_ORDER = ['cortex_septal', 'cortex_outer', 'cortex_dense', 'cortex', 'cortex_residual',
                   'medulla', 'medulla_hc', 'medulla_residual', 'adipose', 'adipose_edge',
                   'fibrosis', 'fibrosis_adipose', 'endothelium_vessel', 'endothelium_edge']
MORPHO_ORDERS = {'morpho_niche_l2': MORPHO_L2_ORDER, 'morpho_niche_l3': MORPHO_L3_ORDER}
MIN_CELLS, MIN_PATCHES, MIN_PATCH_COUNTS = 25, 25, 50_000
print(f'{today} | patch object: {os.path.basename(PATCH_H5AD)}')

2026-09-04 | patch object: thyAging_xenium_patchExpr_7donors_2026-09-04.h5ad


## 1 · The shared gene universe

The patch object holds 5,051 features (v1 panel + the 50 genes of the U48/U51 run); the cell
object 5,043. The scoring universe is their intersection restricted to genes present in **every**
run — so a gene measured only in the infant sections can never drive an age contrast.

In [2]:
patch = ad.read_h5ad(PATCH_H5AD)
with h5py.File(CELL_H5AD, 'r') as f:
    g = f['obs']
    cell_names = pd.Index(np.asarray(g[g.attrs['_index']][:], dtype='S').astype(str))
    cats = np.asarray(g['donor']['categories'][:], dtype='S').astype(str)
    cell_donor = pd.Series(pd.Categorical.from_codes(g['donor']['codes'][:], categories=list(cats)).astype(str), index=cell_names)
    gv = f['var']
    cell_genes = pd.Index(np.asarray(gv[gv.attrs['_index']][:], dtype='S').astype(str))

SHARED = patch.var_names[patch.var['in_v1_panel'].to_numpy()].intersection(cell_genes)
print(f'patch object {patch.n_vars:,} features | cell object {len(cell_genes):,} | shared panel {len(SHARED):,}')
print(f'  in the cell object but not on the v1 panel (the U48/U51 extras): {len(cell_genes.difference(SHARED)):,}')
print(f'  on the v1 panel but not in the cell object: {len(patch.var_names[patch.var["in_v1_panel"].to_numpy()].difference(cell_genes)):,}')
pd.DataFrame({'gene': cell_genes, 'shared_panel': cell_genes.isin(SHARED)}).to_csv(f'{RESULTS}/shared_panel_genes.csv', index=False)

patch object 5,051 features | cell object 5,043 | shared panel 4,993
  in the cell object but not on the v1 panel (the U48/U51 extras): 50
  on the v1 panel but not in the cell object: 8


## 2 · Cell annotations, joined by label

Two side-car CSVs, joined onto the h5ad row order **by cell name** and checked, never by position.

In [3]:
def read_indexed_csv(path, cols, reindex_to, categorical=()):
    """Read `cols` (+ the file's own index column) and reindex onto `reindex_to`."""
    index_col = pd.read_csv(path, nrows=0).columns[0]
    df = pd.read_csv(path, index_col=0, usecols=lambda c: c in cols or c == index_col,
                     dtype={c: 'category' for c in categorical})
    assert df.index.is_unique, f'{os.path.basename(path)}: duplicate index labels'
    missing = [c for c in cols if c not in df.columns]
    assert not missing, f'{os.path.basename(path)}: missing {missing}'
    return df.reindex(reindex_to)


anno_cols = ['taa_l2_comb', 'taa_l3_comb', 'taa_l2_upd', 'taa_l3_upd', 'compart']
morpho_cols = ['patchID', 'morpho_niche_l1', 'morpho_niche_l2', 'morpho_niche_l3', 'morpho_niche_keep']
t0 = time.time()
anno = read_indexed_csv(ANNO_CSV, anno_cols, cell_names, categorical=anno_cols)
morpho = read_indexed_csv(MORPHO_CSV, morpho_cols, cell_names, categorical=morpho_cols[:-1])
obs = pd.concat([anno, morpho], axis=1)
obs['donor'] = cell_donor.to_numpy()
obs['morpho_niche_keep'] = obs['morpho_niche_keep'].fillna(False).astype(bool)
obs['age_group'] = obs['donor'].map(DONOR_AGE)
obs['age_group2'] = obs['age_group'].map(AGE2)
obs['age_years'] = obs['donor'].map(DONOR_AGE_YEARS)
obs['keep'] = (obs['donor'].isin(DONOR_AGE) & obs['morpho_niche_keep']
               & obs['taa_l2_upd'].notna() & obs['taa_l3_upd'].notna() & obs['patchID'].notna())
print(f'annotations joined in {time.time() - t0:.0f}s')
print(f'cells in h5ad                 {len(obs):,}')
print(f'  A45 (dropped)               {int((obs["donor"] == "A45").sum()):,}')
print(f'  no updated cell-type call   {int((obs["taa_l3_upd"].isna() & (obs["donor"] != "A45")).sum()):,} (outside A45)')
print(f'  artifact / off-grid patch   {int((~obs["morpho_niche_keep"]).sum()):,}')
print(f'kept for pseudobulks          {int(obs["keep"].sum()):,}')

# the cell's niche must be the niche of its patch
k = obs['keep'].to_numpy()
p_l2 = patch.obs['morpho_niche_l2'].reindex(obs.loc[k, 'patchID'].astype(str)).to_numpy()
agree = (p_l2 == obs.loc[k, 'morpho_niche_l2'].astype(str).to_numpy()).mean()
print(f'\nniche label of kept cells == niche of their patch: {agree:.4%}')
assert agree > 0.999

annotations joined in 21s
cells in h5ad                 3,531,679
  A45 (dropped)               89,285
  no updated cell-type call   13,936 (outside A45)
  artifact / off-grid patch   112,029


kept for pseudobulks          3,319,282



niche label of kept cells == niche of their patch: 100.0000%


In [4]:
kept = obs.loc[obs['keep']]
print('kept cells per donor x age group:')
print(kept.groupby(['age_group', 'donor'], observed=True).size().rename('n_cells').to_frame().T.to_string())
print('\nkept cells per donor x niche (morpho_niche_l2):')
print(pd.crosstab(kept['donor'], kept['morpho_niche_l2'].astype(str)).reindex(DONOR_ORDER).reindex(columns=MORPHO_L2_ORDER).to_string())
fb = kept[kept['compart'].astype(str) == 'FbAdipo']
print(f'\nfibroblast compartment (compart == FbAdipo): {len(fb):,} kept cells')
print(pd.crosstab(fb['donor'], fb['taa_l3_upd'].astype(str)).reindex(DONOR_ORDER).to_string())

# derived keys: the niche label, masked outside the fibroblast compartment -> fibroblast-only pseudobulks
is_fb = (obs['compart'].astype(str) == 'FbAdipo') & obs['keep']
for key in ['morpho_niche_l2', 'morpho_niche_l3']:
    obs[f'fb_{key}'] = obs[key].astype(str).where(is_fb)
CELL_KEYS = ['morpho_niche_l2', 'morpho_niche_l3', 'taa_l2_upd', 'taa_l3_upd', 'fb_morpho_niche_l2', 'fb_morpho_niche_l3']

kept cells per donor x age group:


age_group   adult                   epaed           lpaed        
donor         A67     A70     A72     U48     U51     Z16     Z17
n_cells    197055  152311  910736  991491  491481  181015  395193

kept cells per donor x niche (morpho_niche_l2):


morpho_niche_l2  cortex_septal  cortex  medulla  medulla_hc  adipose  fibrosis_adipose  fibrosis  vessel
donor                                                                                                   
U51                      40606  350746    90060        4083     1937               368      2274    1407
U48                      46262  659593   253869        8283     5701              2734      6274    8775
Z16                      16240  104172    47632        2774     2130               745      4658    2664
Z17                      48519  143581   178644        9551     4920               680      3267    6031
A67                      37445    8643    95634       18889    24917               614      4034    6879
A72                      44643  279704   399249       19664    31049              8546     37599   90282
A70                       7779    6521    69840        1863    40621              1888      5369   18430



fibroblast compartment (compart == FbAdipo): 60,932 kept cells
taa_l3_upd  Adipocyte  Fb-TGFB+  Fb-adipo  Fb-interlo  Fb-interm  Fb-med  Fb-perilo
donor                                                                              
U51                22        32        33         304        197     502        511
U48                 4        30        18         540        269    1652        866
Z16                 3         1         2         125         43     159        374
Z17                62        14        21         659        347     840       1718
A67               757        22       339        1262       2454     383       4960
A72              2025        58       831        3056       3102    3754       9507
A70              2850        21       342        2697        616    1351      11197


## 3 · One streaming pass over the cell object

`dc.pp.pseudobulk` needs the matrix in memory and the object is 18 GB (832 M non-zeros), so group
sums are accumulated from contiguous 200 k-row chunks, all six keys at once. The accumulator is
checked against `dc.pp.pseudobulk` on the smallest section below.

In [5]:
def group_codes(obs, key, keep):
    """Integer code per row for (donor, key) groups; -1 where the row is excluded."""
    lab = obs[key].astype('object').where(keep & obs[key].notna())
    grp = pd.Series(np.where(lab.isna(), None, obs['donor'].astype(str) + '||' + lab.astype(str)), index=obs.index)
    levels = pd.Index(sorted(grp.dropna().unique()))
    return levels.get_indexer(grp.fillna('__drop__')).astype(np.int64), levels


def stream_pseudobulk(h5ad_path, obs, keys, elem='layers/counts', chunk=200_000, min_cells=MIN_CELLS):
    """Sum raw counts per (donor x group) for several keys in one pass. Returns {key: AnnData}."""
    keep = obs['keep'].to_numpy()
    codes, levels = {}, {}
    for k in keys:
        codes[k], levels[k] = group_codes(obs, k, keep)
    with h5py.File(h5ad_path, 'r') as f:
        gv = f['var']
        var_names = pd.Index(np.asarray(gv[gv.attrs['_index']][:], dtype='S').astype(str))
        dset = sparse_dataset(f[elem])
        n_rows, n_genes = dset.shape
        assert n_rows == len(obs)
        acc = {k: np.zeros((len(levels[k]), n_genes), dtype=np.float64) for k in keys}
        n_cells = {k: np.zeros(len(levels[k]), dtype=np.int64) for k in keys}
        t0 = time.time()
        for start in range(0, n_rows, chunk):
            stop = min(start + chunk, n_rows)
            if not keep[start:stop].any():
                continue
            blk = dset[start:stop]
            for k in keys:
                c = codes[k][start:stop]
                sel = np.flatnonzero(c >= 0)
                if not sel.size:
                    continue
                S = sp.csr_matrix((np.ones(sel.size), (c[sel], sel)), shape=(len(levels[k]), stop - start))
                acc[k] += np.asarray((S @ blk).todense())
                n_cells[k] += np.bincount(c[sel], minlength=len(levels[k]))
            del blk
        print(f'streamed {n_rows:,} rows in {time.time() - t0:.0f}s')
    out = {}
    for k in keys:
        lev = levels[k]
        pb_obs = pd.DataFrame({'donor': [s.split('||')[0] for s in lev], 'group': [s.split('||')[1] for s in lev]},
                              index=[s.replace('||', '_') for s in lev])
        pb_obs['psbulk_cells'] = n_cells[k]
        pb_obs['psbulk_counts'] = acc[k].sum(axis=1)
        a = ad.AnnData(X=sp.csr_matrix(acc[k].astype(np.float32)), obs=pb_obs, var=pd.DataFrame(index=var_names))
        a = a[a.obs['psbulk_cells'] >= min_cells].copy()
        out[k] = a
        print(f'  {k:22s} {a.n_obs:4d} pseudobulks ({len(lev)} before min_cells={min_cells}); '
              f'{a.obs["group"].nunique()} groups x {a.obs["donor"].nunique()} donors')
    return out


def finish(a, level, representation):
    """Common obs/var schema for every stored pseudobulk."""
    a.obs['donor'] = a.obs['donor'].astype(str)
    a.obs['group'] = a.obs['group'].astype(str)
    a.obs['level'] = level
    a.obs['representation'] = representation
    a.obs['age_group'] = a.obs['donor'].map(DONOR_AGE)
    a.obs['age_group2'] = a.obs['age_group'].map(AGE2)
    a.obs['age_years'] = a.obs['donor'].map(DONOR_AGE_YEARS)
    for c in ['donor', 'group', 'level', 'representation', 'age_group', 'age_group2']:
        a.obs[c] = pd.Categorical(a.obs[c].astype(str))
    a.var['shared_panel'] = a.var_names.isin(SHARED)
    a.obs['psbulk_counts_shared'] = np.asarray(a[:, a.var['shared_panel'].to_numpy()].X.sum(axis=1)).ravel()
    return a


pb_cell = stream_pseudobulk(CELL_H5AD, obs, CELL_KEYS)
for k in CELL_KEYS:
    rep = 'fbcells' if k.startswith('fb_') else 'cells'
    pb_cell[k] = finish(pb_cell[k], k.replace('fb_', ''), rep)

streamed 3,531,679 rows in 38s
  morpho_niche_l2          56 pseudobulks (56 before min_cells=25); 8 groups x 7 donors
  morpho_niche_l3          96 pseudobulks (96 before min_cells=25); 14 groups x 7 donors
  taa_l2_upd              188 pseudobulks (201 before min_cells=25); 29 groups x 7 donors
  taa_l3_upd              316 pseudobulks (347 before min_cells=25); 51 groups x 7 donors
  fb_morpho_niche_l2       46 pseudobulks (56 before min_cells=25); 8 groups x 7 donors
  fb_morpho_niche_l3       68 pseudobulks (95 before min_cells=25); 13 groups x 7 donors


In [6]:
# The accumulator is the only hand-written numerics here: check it against decoupler on Z16 (smallest section).
with h5py.File(CELL_H5AD, 'r') as f:
    codes_d = f['obs']['donor']['codes'][:]
z = np.flatnonzero(codes_d == list(cats).index('Z16'))
start, stop = int(z.min()), int(z.max()) + 1
assert stop - start == len(z), 'Z16 rows are not contiguous'
with h5py.File(CELL_H5AD, 'r') as f:
    Xz = sparse_dataset(f['layers/counts'])[start:stop]
az = ad.AnnData(X=Xz, obs=obs.iloc[start:stop][['donor', 'taa_l2_upd', 'keep']].copy(), var=pd.DataFrame(index=cell_genes))
az = az[az.obs['keep'].to_numpy()].copy()
az.obs['taa_l2_upd'] = az.obs['taa_l2_upd'].astype(str)
az.layers['counts'] = az.X.copy()
ref = dc.pp.pseudobulk(az, sample_col='donor', groups_col='taa_l2_upd', mode='sum', layer='counts')
mine = pb_cell['taa_l2_upd']
mine = mine[mine.obs['donor'] == 'Z16']
R = pd.DataFrame(np.asarray(ref.X), index=ref.obs['taa_l2_upd'].astype(str), columns=ref.var_names)
M = pd.DataFrame(np.asarray(mine.X.todense()), index=mine.obs['group'], columns=mine.var_names)
common = R.index.intersection(M.index)
np.testing.assert_array_equal(R.loc[common, mine.var_names].to_numpy(), M.loc[common].to_numpy())
print(f'stream_pseudobulk == dc.pp.pseudobulk on Z16: {len(common)} cell types x {M.shape[1]} genes')
del az, Xz, ref

stream_pseudobulk == dc.pp.pseudobulk on Z16: 26 cell types x 5043 genes


## 4 · Patch pseudobulks

Kept patches (outside the artifact mask, ≥ 1 transcript) summed per donor × niche. `n_cells` is the
number of segmented cells inside those patches, so the two representations can be compared on the
same footing.

In [7]:
def patch_pseudobulk(patch, key, min_patches=MIN_PATCHES, min_counts=MIN_PATCH_COUNTS):
    keep = patch.obs['keep'].to_numpy()
    grp = pd.Series(patch.obs['donor'].astype(str) + '||' + patch.obs[key].astype(str), index=patch.obs_names).where(keep)
    levels = pd.Index(sorted(grp.dropna().unique()))
    codes = levels.get_indexer(grp.fillna('__drop__'))
    sel = np.flatnonzero(codes >= 0)
    S = sp.csr_matrix((np.ones(sel.size, dtype=np.float32), (codes[sel], sel)), shape=(len(levels), patch.n_obs))
    X = (S @ patch.X).tocsr().astype(np.float32)
    pb_obs = pd.DataFrame({'donor': [s.split('||')[0] for s in levels], 'group': [s.split('||')[1] for s in levels]},
                          index=[s.replace('||', '_') for s in levels])
    pb_obs['psbulk_cells'] = np.bincount(codes[sel], minlength=len(levels))          # = number of patches
    pb_obs['n_cells'] = np.bincount(codes[sel], weights=patch.obs['n_cells'].to_numpy()[sel], minlength=len(levels)).astype(int)
    pb_obs['psbulk_counts'] = np.asarray(X.sum(axis=1)).ravel()
    a = ad.AnnData(X=X, obs=pb_obs, var=patch.var[['in_v1_panel', 'v2_only']].copy())
    n0 = a.n_obs
    a = a[(a.obs['psbulk_cells'] >= min_patches) & (a.obs['psbulk_counts'] >= min_counts)].copy()
    print(f'{key}: {a.n_obs} donor x niche pseudobulks ({n0} before the min_patches={min_patches} / min_counts={min_counts:,} filter)')
    return a


pb_patch = {}
for key in ['morpho_niche_l2', 'morpho_niche_l3']:
    pb_patch[key] = finish(patch_pseudobulk(patch, key), key, 'patches')
print()
print(pd.crosstab(pb_patch['morpho_niche_l2'].obs['donor'], pb_patch['morpho_niche_l2'].obs['group']).reindex(DONOR_ORDER).reindex(columns=MORPHO_L2_ORDER).to_string())

morpho_niche_l2: 56 donor x niche pseudobulks (56 before the min_patches=25 / min_counts=50,000 filter)


morpho_niche_l3: 82 donor x niche pseudobulks (96 before the min_patches=25 / min_counts=50,000 filter)

group  cortex_septal  cortex  medulla  medulla_hc  adipose  fibrosis_adipose  fibrosis  vessel
donor                                                                                         
U51                1       1        1           1        1                 1         1       1
U48                1       1        1           1        1                 1         1       1
Z16                1       1        1           1        1                 1         1       1
Z17                1       1        1           1        1                 1         1       1
A67                1       1        1           1        1                 1         1       1
A72                1       1        1           1        1                 1         1       1
A70                1       1        1           1        1                 1         1       1


## 5 · Write, and check the two constructions against each other

At donor × niche resolution the cell pseudobulk and the patch pseudobulk are the same aggregation
minus the transcripts no cell claimed, so their gene profiles must agree closely (Pearson on log
sums). `frac_unassigned` is that missing share.

In [8]:
manifest = []
for key, a in pb_patch.items():
    path = f'{PB_DIR}/pb_patches_{key}.h5ad'
    a.write_h5ad(path)
    manifest.append({'file': os.path.basename(path), 'representation': 'patches', 'level': key, 'n_samples': a.n_obs, 'n_genes': a.n_vars})
for key, a in pb_cell.items():
    rep = a.obs['representation'].iloc[0]
    path = f'{PB_DIR}/pb_{rep}_{key.replace("fb_", "")}.h5ad'
    a.write_h5ad(path)
    manifest.append({'file': os.path.basename(path), 'representation': rep, 'level': key.replace('fb_', ''), 'n_samples': a.n_obs, 'n_genes': a.n_vars})
manifest = pd.DataFrame(manifest)
manifest.to_csv(f'{PB_DIR}/manifest.csv', index=False)
print(f'written to {PB_DIR}')
manifest

written to /nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/data/xenium/pseudobulk_final


,file,representation,level,n_samples,n_genes
0,pb_patches_morpho_niche_l2.h5ad,patches,morpho_niche_l2,56,5051
1,pb_patches_morpho_niche_l3.h5ad,patches,morpho_niche_l3,82,5051
2,pb_cells_morpho_niche_l2.h5ad,cells,morpho_niche_l2,56,5043
3,pb_cells_morpho_niche_l3.h5ad,cells,morpho_niche_l3,96,5043
4,pb_cells_taa_l2_upd.h5ad,cells,taa_l2_upd,188,5043
5,pb_cells_taa_l3_upd.h5ad,cells,taa_l3_upd,316,5043
6,pb_fbcells_morpho_niche_l2.h5ad,fbcells,morpho_niche_l2,46,5043
7,pb_fbcells_morpho_niche_l3.h5ad,fbcells,morpho_niche_l3,68,5043


In [9]:
rows = []
for key in ['morpho_niche_l2', 'morpho_niche_l3']:
    P, Cc = pb_patch[key], pb_cell[key]
    shared_samples = P.obs_names.intersection(Cc.obs_names)
    Xp = np.asarray(P[shared_samples, list(SHARED)].X.todense())
    Xc = np.asarray(Cc[shared_samples, list(SHARED)].X.todense())
    for i, s in enumerate(shared_samples):
        rows.append({'level': key, 'sample': s, 'donor': s.split('_')[0],
                     'tx_counts': Xp[i].sum(), 'cell_counts': Xc[i].sum(),
                     'frac_unassigned': 1 - Xc[i].sum() / Xp[i].sum(),
                     'gene_profile_r': np.corrcoef(np.log1p(Xp[i]), np.log1p(Xc[i]))[0, 1]})
calib = pd.DataFrame(rows)
calib.to_csv(f'{RESULTS}/patch_vs_cell_pseudobulk_{today}.csv', index=False)
print('per donor (morpho_niche_l2 samples):')
print(calib[calib['level'] == 'morpho_niche_l2'].groupby('donor')[['frac_unassigned', 'gene_profile_r']]
      .agg(['mean', 'min', 'max']).reindex(DONOR_ORDER).round(4).to_string())
assert calib['gene_profile_r'].min() > 0.95

per donor (morpho_niche_l2 samples):
      frac_unassigned                 gene_profile_r                
                 mean     min     max           mean     min     max
donor                                                               
U51            0.1245  0.1017  0.1676         0.9916  0.9774  0.9986
U48            0.0100  0.0035  0.0274         0.9980  0.9961  0.9996
Z16            0.0270  0.0103  0.0453         0.9942  0.9880  0.9967
Z17            0.0144  0.0023  0.0344         0.9946  0.9906  0.9973
A67            0.1169  0.0609  0.2092         0.9939  0.9810  0.9984
A72            0.0325  0.0124  0.0581         0.9962  0.9940  0.9980
A70            0.1011  0.0383  0.2490         0.9948  0.9859  0.9984


In [10]:
# depth per group and age group, for the depth checks in the scoring notebooks
depth = []
for a in list(pb_patch.values()) + list(pb_cell.values()):
    d = a.obs.groupby(['representation', 'level', 'group', 'age_group'], observed=True).agg(
        n_donors=('donor', 'nunique'), median_cells=('psbulk_cells', 'median'), median_counts=('psbulk_counts_shared', 'median')).reset_index()
    depth.append(d)
depth = pd.concat(depth, ignore_index=True)
depth.to_csv(f'{RESULTS}/pseudobulk_depth_{today}.csv', index=False)

# cell-type composition per donor x niche (composition control for the scoring notebooks)
comp = []
for nk in ['morpho_niche_l2', 'morpho_niche_l3']:
    for ck in ['taa_l2_upd', 'taa_l3_upd']:
        t = kept.groupby(['donor', nk, ck], observed=True).size().rename('n').reset_index()
        t.columns = ['donor', 'niche', 'ctype', 'n']
        t['frac'] = t['n'] / t.groupby(['donor', 'niche'])['n'].transform('sum')
        t['niche_level'], t['ctype_level'] = nk, ck
        comp.append(t)
comp = pd.concat(comp, ignore_index=True)
comp.to_csv(f'{RESULTS}/composition_by_niche_{today}.csv', index=False)
fbfrac = comp[(comp['niche_level'] == 'morpho_niche_l2') & (comp['ctype_level'] == 'taa_l2_upd') & comp['ctype'].isin(['Fb', 'Adipocyte'])]
print('fibroblast + adipocyte fraction (%) per donor x niche:')
print((fbfrac.groupby(['donor', 'niche'])['frac'].sum().unstack('niche').reindex(DONOR_ORDER).reindex(columns=MORPHO_L2_ORDER) * 100).round(2).to_string())

fibroblast + adipocyte fraction (%) per donor x niche:
niche  cortex_septal  cortex  medulla  medulla_hc  adipose  fibrosis_adipose  fibrosis  vessel
donor                                                                                         
U51             0.81    0.15     0.75        0.37     1.03              1.63      0.26    1.35
U48             0.64    0.11     0.76        0.24     0.72              0.80      1.59    3.05
Z16             1.09    0.07     0.62        0.11     0.56              2.55      1.27    2.40
Z17             1.94    0.18     0.98        0.28     2.34              8.97      6.03    5.11
A67             5.33    4.21     2.68        0.74    14.24             19.22     17.65   10.63
A72             3.40    0.46     2.12        0.52     8.93              6.39      4.54    6.57
A70             8.47   13.39     6.54        1.99    20.49             37.92     22.41   14.60


In [11]:
print('pseudobulk files:')
for f_ in sorted(os.listdir(PB_DIR)):
    print(f'  {f_:45s} {os.path.getsize(f"{PB_DIR}/{f_}") / 1e6:6.1f} MB')
print(f'\ntables -> {RESULTS}')

pseudobulk files:
  manifest.csv                                     0.0 MB
  pb_cells_morpho_niche_l2.h5ad                    2.4 MB
  pb_cells_morpho_niche_l3.h5ad                    3.8 MB
  pb_cells_taa_l2_upd.h5ad                         6.8 MB
  pb_cells_taa_l3_upd.h5ad                        10.8 MB
  pb_fbcells_morpho_niche_l2.h5ad                  1.6 MB
  pb_fbcells_morpho_niche_l3.h5ad                  2.1 MB
  pb_patches_morpho_niche_l2.h5ad                  2.5 MB
  pb_patches_morpho_niche_l3.h5ad                  3.5 MB

tables -> /nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/results/xenium/final/02_pseudobulks
